# Studio sulla percezione dell'Intelligenza Artificiale in ambito professionale

**Obiettivo principale:**
Il presente progetto mira a condurre una ricerca per rispondere alle seguenti domande chiave relative all'adozione e alla percezione dell'Intelligenza Artificiale (_Atificial Intelligence_ - AI) nel mondo del lavoro:
1. La discussione sull'AI è prevalentemente positiva o negativa nelle diverse professioni?
2. Le percezioni dell'AI cambiano tra professioni differenti?
3. Chi vede l'AI come minaccia e chi come opportunità?
4. Utenti che condividono *stance* sull'AI creano **echo-chambers** oppure c'è un fenomeno di **cross-pollination**?
5. Gruppi di professioni differenti condividono discussioni oppure si trovano in bolle separate?

**Metodologia di Analisi:**
L'analisi sarà strutturata in diverse fasi metodologiche interconnesse:
*   Raccolta di dati provenienti dai social media (Reddit - dump Arctic Shift).
*   Analisi delle *stance* / sentiment per ogni utente, aggregando le stance dei commenti dell'utente.
*   Realizzazione di un'analisi di rete (*network analysis*) per verificare la presenza di echo-chambers e cross-pollination.
*   Interpretazione dei risultati ed estrazione degli insights

**Fonti di Raccolta Dati:**
1.  **Piattaforma:** I dati saranno raccolti dal social media **Reddit** utilizzando i dump forniti da **Arctic Shift**.
2.  **Contesto:** Verranno analizzati i commenti di post che trattano argomenti relativi all'Intelligenza Artificiale provenienti da subreddit specifici legati a professioni ben definite.

**Profili Professionali di Studio:**
Sono state selezionate le seguenti professionalità in quanto rappresentano settori chiave e sono direttamente influenzate dalle campagne marketing dei principali produttori di modelli IA (es. prodotti che mirano a servizi specifici come "Claude Code", o strumenti per il design, ecc.):
*   Programmatori (`r/AskProgramming`, `r/webdev`)
*   Designer Grafici/Artisti (`r/graphic_design`, `r/artistlounge`)
*   Specialisti di Marketing e Copywriting (`r/marketing`, `r/copywriting`)

**Parole Chiave di Ricerca (Keywords):**
Il campionamento dei dati si concentrerà su post contenenti i seguenti termini:

| Keyword                 | Significato                                               | Motivazione                                                    |
| ----------------------- | --------------------------------------------------------- | -------------------------------------------------------------- |
| AI                      | Acronimo di Artificial Intelligence                       | Termine più comune e trasversale                               |
| artificial intelligence | Forma estesa di AI                                        | Aumenta la copertura delle discussioni                         |
| machine learning        | Tecniche che permettono ai sistemi di apprendere dai dati | Molto usato in contesti tecnici e professionali                |
| ChatGPT                 | Assistente AI basato su LLM                               | Simbolo principale del dibattito sull'AI                       |
| Copilot                 | Assistente AI per attività professionali                  | Diffuso nell'automazione del lavoro                            |
| Midjourney              | Generatore di immagini tramite AI                         | Rilevante nelle professioni creative                           |
| Stable Diffusion        | Modello open-source per generare immagini                 | Centrale nelle discussioni su arte                             |
| automation              | Automazione di attività lavorative                        | Direttamente collegata all'impatto sul lavoro                  |
| job loss                | Perdita di posti di lavoro                                | Individua discussioni sui rischi occupazionali                 |
| replace                 | Sostituire persone o mansioni                             | Spesso usato nei dibattiti sulla sostituzione del lavoro umano |
| workflow                | Flusso di lavoro professionale                            | Individua discussioni sui cambiamenti operativi                |
| generative AI           | AI capace di generare contenuti                           | Tecnologia al centro delle trasformazioni professionali        |
| LLM                     | Large Language Model                                      | Termine comune nelle discussioni su ChatGPT e simili           |
| prompt                  | Istruzione fornita a un modello AI                        | Rappresenta una competenza pratica nell'uso dell'AI            |


In [5]:
# Setup e import
import os
import sys

module_path = os.path.abspath(os.path.join('../src/data_retrive.py'))

if module_path not in sys.path:
    sys.path.append(module_path)
from pathlib import Path
from src.data_retrive import runSubredditPipeline
import pandas as pd
from IPython.display import display


# Step 1: Raccolta e Pulizia dei Dati

La raccolta e la pulizia dei dati avvengono secondo i seguenti passaggi:
1. Vengono processati i dump files dei post dei vari subreddit considerati
2. Per ogni subreddit, si filtrano i post per le keywords
3. Si crea una lista di id di post considerati
4. Vengono puliti i dati dei post di modo da rimuovere post non utili alla ricerca (approfondimento più avanti in questo step)
5. Vengono processati i dump files dei commenti dei vari subreddit considerati
6. Per ogni subreddit, si filtrano i commenti che appartengono ai post selezionati precedentemente per il dato subreddit e che sono di primo livello
7. Vengono salvati tutti i commenti processati per ogni subreddit associando ad ognuno di essi il subreddit da cui è stato recuperato
8. Prima della stance analysis, i post e i commenti raccolti vengono ulteriormente puliti ed esplorati per garantire che siano completi, coerenti e rappresentativi del fenomeno oggetto di studio

## Subreddit Considerati
Subreddit considerati nella raccolta dei dati:
- r/AskProgramming
- r/webdev
- r/marketing
- r/graphic_design
- r/artistlounge
- r/copywriting

Sono stati individuati i seguenti subreddit in quanto condividono le seguenti caratteristiche:
- Permettono discussioni su AI
- Mantengono le discussioni legate ad argomenti relativi alle professioni specifiche del subreddit
- Popolate principalmente da professionisti degli ambiti relativi agli argomenti del subreddit

Per ciascuno di questi subreddit vengono processati sia i dump dei post (passaggio 1) sia, in un secondo momento, i dump dei commenti (passaggio 5).

## Strumento di Raccolta Dati
La raccolta dei dump viene effettuata mediante i file forniti da Arctic Shift, scelti in virtù della loro maggiore efficienza rispetto alle API e della complessità associata all'accesso alle API ufficiali di Reddit, ritenuta non proporzionata agli obiettivi e alla scala del presente progetto.

Ulteriori modalità di accesso ai dati, quali gli endpoint JSON pubblici, non risultano più disponibili a partire da giugno 2026. Il loro utilizzo richiede infatti la registrazione di un'applicazione e l'ottenimento di specifiche autorizzazioni da parte di Reddit, comportando un processo sostanzialmente analogo a quello previsto per l'accesso alle API ufficiali.

Per ogni subreddit, Arctic Shift fornisce due dump distinti in formato `.jsonl`, uno per i post e uno per i commenti, processati separatamente secondo l'ordine descritto sopra: prima i post (passaggi 1-4), poi i commenti (passaggi 5-7).

### Limiti Temporali
Il dibattito pubblico sull'AI ha avuto un'accelerazione significativa dopo il lancio di ChatGPT a fine 2022.
Si presuppone quindi che precedentemente al 2023 vi erano pochi post rilevanti, si prosegue poi nel 2023 in cui iniziano dibatti più intensi ed interessanti, ma ancora molto caotici.
Partendo dal 2024 il dibattito si può definire più "maturo" con posizioni più solide tra le professioni.

L'anno 2024 risulta quindi quello più interessante per il fenomeno ricercato.

La raccolta dei dati viene dunque limitata ad un intervallo temporale definito: tra il **1 gennaio 2024** e il **31 dicembre 2024**. Questo limite viene applicato uniformemente a tutti i subreddit analizzati, riducendo il rischio di bias legati ad eventi contingenti che potrebbero aver influenzato il dibattito sull'AI in periodi specifici.


In [2]:
# Setup per Raccolta Dati

# Funzioni Ausiliarie

def _find_data_file(folderPath: Path, suffix: str) -> Path | None:
    """Trova il file .jsonl il cui nome termina con il suffisso indicato (es. '_posts', '_comments')."""
    matches = sorted(Path(folderPath).glob(f"*{suffix}.jsonl"))
    return matches[0] if matches else None

def _processed_csv_path(jsonlFile: Path) -> str:
    """Percorso del csv processato: stessa cartella e stesso nome del .jsonl, con suffisso '_processed'."""
    return str(jsonlFile.with_name(jsonlFile.stem + "_processed.csv"))

def _save_dataframe(df: pd.DataFrame, path: str, fields: list[str]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, columns=fields, index=False, encoding="utf-8")



In [3]:
import json


# STEP 1 - Processamento del dump dei post di un subreddit

def processPostDumps(subredditFolder: str) -> pd.DataFrame:
    """
    Step 1: legge il dump .jsonl dei post di UN singolo subreddit
    (subredditFolder) e lo raccoglie in un DataFrame, senza applicare alcun filtro.
    """
    subredditFolder = Path(subredditFolder)
    postsFile = _find_data_file(subredditFolder, "_posts")
    if postsFile is None:
        raise FileNotFoundError(f"Nessun file di post trovato in {subredditFolder}")

    print(f"- Processing posts file: {postsFile.name}")
    rows = []
    for post in readJSONL(postsFile):
        rows.append({
            "id": post.get("id", ""),
            "author": post.get("author", ""),
            "subreddit": post.get("subreddit", subredditFolder.name),
            "title": post.get("title", "") or "",
            "selftext": post.get("selftext", "") or "",
        })

    df_posts = pd.DataFrame(rows, columns=["id", "author", "subreddit", "title", "selftext"])
    print(f"[STEP 1] Post totali raccolti: {len(df_posts):,}")
    return df_posts

def readJSONL(path: Path):
    """Legge un file .jsonl riga per riga e restituisce i dict corrispondenti."""
    with open(path, "rb") as f:
        for line in f:
            line = line.decode("utf-8", errors="replace").strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                print(f"Errore nel parsing della riga in {path.name}")
                continue



## Filtraggio dei Post (Passaggio 2)
Il filtraggio dei post, applicato durante la processazione del dump di ciascun subreddit, si basa su due criteri applicati congiuntamente: la presenza di keywords rilevanti e l'appartenenza all'intervallo temporale definito. Un post viene incluso nel dataset solo se soddisfa entrambi i criteri.

### Filtraggio per Keywords
La raccolta dei dati viene limitata ai contenuti rilevanti per il tema dell'intelligenza artificiale tramite un filtraggio per keywords.
Vengono inclusi nel dataset esclusivamente i post che contengono almeno una delle keywords definite.

#### Keywords Scelte

```
AI
artificial intelligence
machine learning
ChatGPT
Copilot
Midjourney
Stable Diffusion
automation
job loss
replace
workflow
generative AI
LLM
prompt
```

In [4]:
import re

# STEP 2 - Filtraggio dei post per keywords

# Keywords da cercare (case-insensitive, parola intera o sottostringa)
KEYWORDS = [
    "AI",
    "artificial intelligence",
    "machine learning",
    "ChatGPT",
    "Copilot",
    "Midjourney",
    "Stable Diffusion",
    "automation",
    "job loss",
    "replace",
    "workflow",
    "generative AI",
    "LLM",
    "prompt",
]

# Pattern regex compilato una volta sola per performance
_keyword_pattern = re.compile(
    r'(?<!\w)(' + '|'.join(re.escape(kw) for kw in KEYWORDS) + r')(?!\w)',
    re.IGNORECASE
)

def filterPostsByKeywords(df_posts: pd.DataFrame) -> pd.DataFrame:
    """
    Step 2: filtra i post del subreddit tenendo solo quelli che contengono
    almeno una keyword (in titolo o corpo). Aggiunge la colonna 'matched_keywords'.
    """

    combined = (df_posts["title"].fillna("") + " " + df_posts["selftext"].fillna(""))
    is_match = combined.apply(textMatches)

    df_filtered = df_posts.loc[is_match].copy()
    df_filtered["matched_keywords"] = combined.loc[is_match].apply(findKeywords)

    print(f"[STEP 2] Post trovati per keyword: {len(df_posts):,} -> {len(df_filtered):,}")

    return df_filtered.reset_index(drop=True)

def textMatches(text: str) -> bool:
    """True se almeno una keyword è presente nel testo."""
    return bool(text and _keyword_pattern.search(text))

def findKeywords(text: str) -> str:
    """Restituisce le keyword trovate nel testo, separate da '|'."""
    if not text:
        return ""
    found = set(m.group(0).lower() for m in _keyword_pattern.finditer(text))
    return "|".join(sorted(found))



## Creazione della Lista di ID (Passaggio 3)
Per ogni subreddit, gli id dei post che superano il filtraggio per keywords e per intervallo temporale vengono raccolti in una lista.
Questa lista costituisce il riferimento con cui, nel passaggio 6, verranno selezionati i commenti di primo livello appartenenti a ciascun post.


In [5]:
#  STEP 3 - Lista degli id dei post considerati

def getConsideredPostIds(df_posts_filtered: pd.DataFrame) -> set[str]:
    """
    Step 3: costruisce l'insieme degli id (formato "t3_xxxxx") dei post del
    subreddit che hanno superato il filtro keyword. Verrà usato per
    selezionare i commenti pertinenti nello step 6.
    """
    considered_ids = {f"t3_{post_id}" for post_id in df_posts_filtered["id"]}
    print(f"[STEP 3] Id di post considerati: {len(considered_ids):,}")
    return considered_ids


## Pulizia dei Post (Passaggio 4)
Prima di passare alla raccolta dei commenti, i post selezionati vengono sottoposti a una pulizia intermedia, volta a scartare quelli non utilizzabili ai fini della ricerca. Questa pulizia ha l'obiettivo di garantire che le informazioni utilizzate nelle fasi successive siano complete, coerenti e rappresentative del fenomeno oggetto di studio.

### Feature Fondamentali
L'imputazione dei valori mancanti non è una soluzione applicabile nel presente caso di studio: le feature la cui integrità risulta fondamentale non possono essere ricostruite o stimate a partire dalle altre istanze disponibili, e la loro assenza rende l'osservazione inutilizzabile.

Per i post vengono effettuati esclusivamente i controlli necessari alle successive fasi dell'analisi. La pipeline di pulizia prevede i seguenti passaggi:
1. Rimozione dei post con contenuto cancellato/rimosso (selftext o author in DELETED_MARKERS)
2. Rimozione dei post pubblicati da bot (es. AutoModerator)
3. Rimozione dei post duplicati (stesso id)

#### Il Caso "AutoModerator"
L'account **AutoModerator** rappresenta un bot ufficiale della piattaforma Reddit incaricato di svolgere attività automatiche di moderazione, quali la pubblicazione di messaggi informativi, l'applicazione delle regole della comunità e la rimozione automatica di contenuti.

Poiché l'obiettivo della ricerca consiste nell'analizzare le opinioni espresse dagli utenti appartenenti alle diverse professioni, i contenuti pubblicati da AutoModerator vengono esclusi dal dataset. Tali messaggi non rappresentano infatti opinioni umane e la loro inclusione introdurrebbe osservazioni prive di significato rispetto agli obiettivi dello studio.


In [6]:

#  STEP 4 - Pulizia dei post

# Autori "bot" da escludere
BOT_AUTHORS = {"AutoModerator"}

# Valori che indicano contenuto non più disponibile
DELETED_MARKERS = {"[deleted]", "[removed]"}

def cleanPosts(df_posts_filtered: pd.DataFrame, subredditFolder: str) -> pd.DataFrame:
    """
    Step 4: rimuove dai post filtrati per keyword quelli non utili alla ricerca:
    - post con contenuto cancellato/rimosso (selftext o author in DELETED_MARKERS)
    - post pubblicati da bot (es. AutoModerator)
    - post duplicati (stesso id)
    Il risultato viene salvato in un csv accanto al file .jsonl dei post.
    """
    df = df_posts_filtered.copy()
    n_before = len(df)

    is_deleted = df["selftext"].isin(DELETED_MARKERS) | df["author"].isin(DELETED_MARKERS)
    is_bot = df["author"].isin(BOT_AUTHORS)

    df_no_deleted_bot = df.loc[~is_deleted & ~is_bot]
    df_clean = df_no_deleted_bot.drop_duplicates(subset="id")
    n_duplicates = len(df_no_deleted_bot) - len(df_clean)

    print(f"[STEP 4] Pulizia post: {n_before:,} -> {len(df_clean):,} "
          f"(rimossi: {is_deleted.sum():,} cancellati/rimossi, {is_bot.sum():,} bot, "
          f"{n_duplicates:,} duplicati)")

    df_clean = df_clean.reset_index(drop=True)
    savePosts(df_clean, subredditFolder)

    return df_clean



In [7]:
# Salvataggio su File

#  CAMPI CSV

POST_FIELDS = [
    "id",  # ID univoco del post (es. "abc123")
    "author",  # Username autore
    "subreddit",  # Nome subreddit (senza r/)
    "title",  # Titolo del post
    "selftext",  # Testo del corpo del post
    "matched_keywords",  # Keywords trovate nel testo (debug/analisi)
]

COMMENT_FIELDS = [
    "id",  # ID univoco del commento
    "link_id",  # ID del post padre (formato "t3_xxxxx")
    "subreddit",  # Nome subreddit da cui è stato recuperato il commento
    "author",  # Username autore
    "body",  # Testo del commento
]

# Funzioni Ausiliarie
def _processed_csv_path(jsonlFile: Path) -> str:
    """Percorso del csv processato: stessa cartella e stesso nome del .jsonl, con suffisso '_processed'."""
    return str(jsonlFile.with_name(jsonlFile.stem + "_processed.csv"))


def _save_dataframe(df: pd.DataFrame, path: str, fields: list[str]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, columns=fields, index=False, encoding="utf-8")


# Metodo di Salvataggio
def savePosts(df_posts: pd.DataFrame, subredditFolder: str):
    """Salva i post processati in un csv nella cartella del subreddit, con lo
    stesso nome del file .jsonl dei post e suffisso '_processed'."""
    subredditFolder = Path(subredditFolder)
    postsFile = _find_data_file(subredditFolder, "_posts")
    if postsFile is None:
        raise FileNotFoundError(f"Nessun file di post trovato in {subredditFolder}")

    outPath = _processed_csv_path(postsFile)
    _save_dataframe(df_posts, outPath, POST_FIELDS)
    print(f"[SAVE] {len(df_posts):,} post salvati in {outPath}")

## Raccolta dei Commenti (Passaggi 5-7)
Per ogni subreddit viene processato il relativo dump dei commenti. Vengono filtrati esclusivamente i commenti di **primo livello** (cioè risposte dirette al post, non ad altri commenti) il cui post di appartenenza compare nella lista di id costruita al passaggio 3, tra quelli sopravvissuti alla pulizia del passaggio 4. A differenza dei post, questi commenti vengono inclusi indipendentemente dal loro contenuto testuale, in quanto considerati contestualmente rilevanti per il tema della discussione: se il post che li ospita ha superato il filtraggio per keywords, il commento eredita questa rilevanza.

Ogni commento così selezionato viene infine salvato insieme a un'etichetta che indica il subreddit da cui è stato recuperato, in modo da poterlo ricollegare in seguito alla comunità professionale di appartenenza. I dati filtrati (post e commenti) vengono salvati in file CSV.

In [8]:


#  STEP 5 - Processamento del dump dei commenti di un subreddit

def processCommentDumps(subredditFolder: str) -> pd.DataFrame:
    """
    Step 5: legge il dump .jsonl dei commenti di UN singolo subreddit
    e lo raccoglie in un DataFrame, senza applicare alcun filtro.
    """
    subredditFolder = Path(subredditFolder)
    commentsFile = _find_data_file(subredditFolder, "_comments")
    if commentsFile is None:
        raise FileNotFoundError(f"Nessun file di commenti trovato in {subredditFolder}")

    print(f"- Processing comments file: {commentsFile.name}")
    rows = []
    for comment in readJSONL(commentsFile):
        rows.append({
            "id": comment.get("id", ""),
            "link_id": comment.get("link_id", "") or "",
            "parent_id": comment.get("parent_id", "") or "",
            "subreddit": comment.get("subreddit", subredditFolder.name),
            "author": comment.get("author", ""),
            "body": comment.get("body", "") or "",
        })

    df_comments = pd.DataFrame(
        rows, columns=["id", "link_id", "parent_id", "subreddit", "author", "body"]
    )
    print(f"[STEP 5] Commenti totali raccolti: {len(df_comments):,}")
    return df_comments


#  STEP 6 - Filtraggio dei commenti sui post selezionati

def filterCommentsByPosts(df_comments: pd.DataFrame, considered_post_ids: set[str]) -> pd.DataFrame:
    """
    Step 6: filtra i commenti che sono:
    - di primo livello (risposta diretta al post, parent_id == link_id)
    - relativi a un post che ha superato il filtro keyword (STEP 3)
    """
    is_first_level = df_comments["parent_id"] == df_comments["link_id"]
    is_considered = df_comments["link_id"].isin(considered_post_ids)

    df_filtered = df_comments.loc[is_first_level & is_considered].copy()

    print(f"[STEP 6] Commenti trovati: {len(df_comments):,} -> {len(df_filtered):,}")

    return df_filtered.drop(columns=["parent_id"]).reset_index(drop=True)


#  STEP 7 - Salvataggio dei commenti processati

def saveComments(df_comments_filtered: pd.DataFrame, subredditFolder: str) -> pd.DataFrame:
    """
    Step 7: salva i commenti processati del subreddit in un csv nella stessa
    cartella del file .jsonl dei commenti, con lo stesso nome e suffisso '_processed'.
    """
    subredditFolder = Path(subredditFolder)
    commentsFile = _find_data_file(subredditFolder, "_comments")
    if commentsFile is None:
        raise FileNotFoundError(f"Nessun file di commenti trovato in {subredditFolder}")

    outPath = _processed_csv_path(commentsFile)
    _save_dataframe(df_comments_filtered, outPath, COMMENT_FIELDS)
    print(f"[SAVE] {len(df_comments_filtered):,} commenti salvati in {outPath}")
    return df_comments_filtered



## Pulizia dei Commenti (Passaggio 8)
Prima della stance analysis è necessario:
- eliminare bot di moderazione
- rimuovere dati corrotti o inutilizzabili
- contare post e commenti per subreddit
- verificare che i dati siano equilibrati

Il filtraggio applicato in fase di raccolta (passaggi 5-7) verifica solo l'appartenenza al post corretto e il livello del commento, non ancora la qualità o la completezza del contenuto: anche i commenti raccolti richiedono quindi una pulizia dedicata, analoga a quella già effettuata per i post, prima di procedere con la stance analysis.

### Feature Fondamentali
Il contenuto testuale dei commenti è l'informazione principale utilizzata per la classificazione delle opinioni e, come per i post, non può essere ricostruito o stimato a partire dalle altre istanze disponibili in caso di assenza.

Le feature la cui integrità risulta fondamentale per i commenti sono le seguenti:

| Feature   | Motivazione                                                                                |
| --------- | -------------------------------------------------------------------------------------------- |
| `link_id` | Identificativo del post padre, necessario per associare il commento al relativo subreddit.  |
| `author`  | Identificativo dell'autore, necessario per la costruzione della rete degli utenti.          |
| `body`    | Contenuto testuale utilizzato nella stance analysis.                                        |

Il DataFrame dei commenti costituisce la principale fonte informativa del progetto, poiché ogni classificazione di stance viene effettuata sul contenuto testuale dei commenti e non sui post. Per questo motivo vengono eseguiti controlli più rigorosi rispetto a quelli previsti per i post.

La pipeline di pulizia prevede i seguenti passaggi:
1. eliminazione dei commenti privi di `link_id`, poiché non possono essere associati ad alcun post e, di conseguenza, al relativo subreddit
2. eliminazione dei commenti con `body` nullo, vuoto oppure contenente esclusivamente i valori speciali `[removed]` o `[deleted]`
3. eliminazione dei commenti il cui autore risulta nullo oppure identificato come `[deleted]`
4. eliminazione dei commenti pubblicati dall'account **AutoModerator**
5. eventuale rimozione di duplicati

#### Il caso `[removed]`
Quando un commento viene rimosso dai moderatori oppure dagli strumenti automatici di moderazione di Reddit, il contenuto originale non è più disponibile e il campo `body` assume il valore `[removed]`.

In tali casi il testo non è più recuperabile e il commento non può essere sottoposto alla stance analysis. Viene quindi eliminato dal dataset.

#### Il caso `[deleted]`
Quando un utente elimina volontariamente il proprio account oppure cancella il commento, Reddit sostituisce il nome dell'autore con il valore `[deleted]`. Se anche il contenuto del commento risulta sostituito da `[deleted]`, il commento viene eliminato poiché non contiene alcuna informazione utile.

Qualora il testo del commento sia ancora disponibile ma l'autore risulti `[deleted]`, viene ugualmente eliminato in quanto l'identità dell'autore rappresenta un'informazione indispensabile per le successive analisi sulle interazioni tra utenti.


In [9]:

#  STEP 8 - Pulizia ed esplorazione finale, pre stance analysis

def cleanAndExploreFinalData(
    df_posts: pd.DataFrame, df_comments: pd.DataFrame, subredditFolder: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Step 8: pulizia ed esplorazione finale di post e commenti del subreddit
    prima della stance analysis, per garantire che i dati siano completi,
    coerenti e rappresentativi del fenomeno studiato.

    Pulizia dei commenti:
      1. rimozione dei commenti privi di link_id (non associabili a un post)
      2. rimozione dei commenti con body nullo, vuoto o in DELETED_MARKERS
      3. rimozione dei commenti con author nullo o "[deleted]"
      4. rimozione dei commenti pubblicati da bot (es. AutoModerator)
      5. rimozione di eventuali duplicati (stesso id)

    Il risultato viene salvato in due csv nella cartella del subreddit,
    accanto ai rispettivi file .jsonl (stesso nome + '_processed').
    """
    # --- Pulizia commenti ---
    df_c = df_comments.copy()
    n_before = len(df_c)

    has_link_id = df_c["link_id"].notna() & (df_c["link_id"] != "")
    body = df_c["body"]
    has_body = body.notna() & (body.str.strip() != "") & (~body.isin(DELETED_MARKERS))
    author = df_c["author"]
    has_author = author.notna() & (author != "[deleted]")
    is_not_bot = ~author.isin(BOT_AUTHORS)

    df_comments_clean = df_c.loc[has_link_id & has_body & has_author & is_not_bot]
    df_comments_clean = df_comments_clean.drop_duplicates(subset="id").reset_index(drop=True)

    print(f"[STEP 8] Pulizia commenti: {n_before:,} -> {len(df_comments_clean):,}")

    print(f"[STEP 8] Post finali: {len(df_posts):,}")
    print(f"[STEP 8] Commenti finali: {len(df_comments_clean):,}")

    savePosts(df_posts, subredditFolder)
    saveComments(df_comments_clean, subredditFolder)

    return df_posts, df_comments_clean


## Applicazione dell'Intera Pipeline
Si applicano ora tutte le funzioni definite per i vari passaggi in un'unica pipeline che completerà il processo di raccolta e pulizia dei dati.

La pipeline è definita per processare un subreddit alla volta, questo significa che dovrà essere eseguita per ogni subreddit separatamente.

In [10]:
def runSubredditPipeline(subredditFolder: str):

    df_posts_raw = processPostDumps(subredditFolder)
    df_posts_filtered = filterPostsByKeywords(df_posts_raw)
    considered_post_ids = getConsideredPostIds(df_posts_filtered)
    df_posts_clean = cleanPosts(df_posts_filtered, subredditFolder)

    df_comments_raw = processCommentDumps(subredditFolder)
    df_comments_filtered = filterCommentsByPosts(df_comments_raw, considered_post_ids)
    saveComments(df_comments_filtered, subredditFolder)

    df_posts_final, df_comments_final = cleanAndExploreFinalData(
        df_posts_clean, df_comments_filtered, subredditFolder
    )

    print("\nDone :>")
    return df_posts_final, df_comments_final

rawDataFolder = r"D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\raw"
for folder in sorted(entry for entry in Path(rawDataFolder).iterdir() if entry.is_dir()):
    print(f"\n===== {folder.name} =====")
    runSubredditPipeline(str(folder))



===== artistlounge =====
- Processing posts file: r_artistlounge_posts.jsonl
[STEP 1] Post totali raccolti: 24,418
[STEP 2] Post trovati per keyword: 24,418 -> 1,323
[STEP 3] Id di post considerati: 1,323
[STEP 4] Pulizia post: 1,323 -> 821 (rimossi: 425 cancellati/rimossi, 77 bot, 0 duplicati)
[SAVE] 821 post salvati in D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\raw\artistlounge\r_artistlounge_posts_processed.csv
- Processing comments file: r_artistlounge_comments.jsonl
[STEP 5] Commenti totali raccolti: 266,494
[STEP 6] Commenti trovati: 266,494 -> 9,397
[SAVE] 9,397 commenti salvati in D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\raw\artistlounge\r_artistlounge_comments_processed.csv
[STEP 8] Pulizia commenti: 9,397 -> 8,205
[STEP 8] Post finali: 821
[STEP 8] Commenti finali: 8,205
[SAVE] 821 post salvati in D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\raw\artistlounge\r_artistlounge_posts_processed.csv
[SAVE] 8,205 commenti salvati in D:\UNI\Corsi\Social Media 

### Aggregazione dei Dati
Si aggregano tutti i dati per i post nel file: all_posts.csv e tutti i dati per i commenti nel file: all_comments.csv

In [11]:
# Aggregazione di tutti i post e commenti processati in due file unici

RAW_DATA_FOLDER = Path(r"D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\raw")
DATA_FOLDER = Path(r"D:\UNI\Corsi\Social Media Mining\Progetto SMM\data")

def aggregateProcessedFiles(suffix: str) -> pd.DataFrame:
    """Unisce in un unico DataFrame tutti i csv processati (es. '_posts_processed.csv')
    trovati nelle sottocartelle di RAW_DATA_FOLDER, uno per ogni subreddit."""
    csvFiles = sorted(RAW_DATA_FOLDER.glob(f"*/*{suffix}"))
    if not csvFiles:
        raise FileNotFoundError(f"Nessun file '*{suffix}' trovato in {RAW_DATA_FOLDER}")

    dfs = []
    for csvFile in csvFiles:
        print(f"- Aggregazione di {csvFile.relative_to(RAW_DATA_FOLDER)}")
        dfs.append(pd.read_csv(csvFile))

    return pd.concat(dfs, ignore_index=True)

# Aggregazione dei post
all_posts_df = aggregateProcessedFiles("_posts_processed.csv")
allPostsPath = DATA_FOLDER / "all_posts.csv"
all_posts_df.to_csv(allPostsPath, index=False, encoding="utf-8")
print(f"[AGGREGAZIONE] {len(all_posts_df):,} post salvati in {allPostsPath}")

# Aggregazione dei commenti
all_comments_df = aggregateProcessedFiles("_comments_processed.csv")
allCommentsPath = DATA_FOLDER / "all_comments.csv"
all_comments_df.to_csv(allCommentsPath, index=False, encoding="utf-8")
print(f"[AGGREGAZIONE] {len(all_comments_df):,} commenti salvati in {allCommentsPath}")


- Aggregazione di artistlounge\r_artistlounge_posts_processed.csv
- Aggregazione di askprogramming\r_AskProgramming_posts_processed.csv
- Aggregazione di copywriting\r_copywriting_posts_processed.csv
- Aggregazione di graphicdesign\r_graphic_design_posts_processed.csv
- Aggregazione di marketing\r_marketing_posts_processed.csv
- Aggregazione di webdev\r_webdev_posts_processed.csv
[AGGREGAZIONE] 8,075 post salvati in D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\all_posts.csv
- Aggregazione di artistlounge\r_artistlounge_comments_processed.csv
- Aggregazione di askprogramming\r_AskProgramming_comments_processed.csv
- Aggregazione di copywriting\r_copywriting_comments_processed.csv
- Aggregazione di graphicdesign\r_graphic_design_comments_processed.csv
- Aggregazione di marketing\r_marketing_comments_processed.csv
- Aggregazione di webdev\r_webdev_comments_processed.csv
[AGGREGAZIONE] 54,591 commenti salvati in D:\UNI\Corsi\Social Media Mining\Progetto SMM\data\all_comments.csv


In [6]:
postsPath = r"../data/all_posts.csv"
commentsPath = r"../data/all_comments.csv"

posts_clean_df = pd.read_csv(postsPath)
comments_clean_df = pd.read_csv(commentsPath)


print(f"\nPosts file:\n {posts_clean_df.head()}")
print(f"\nComments file:\n {comments_clean_df.head()}")

print(f"\n\nPosts DataFrame shape: {posts_clean_df.shape}")
print(f"Comments DataFrame shape: {comments_clean_df.shape}")


Posts file:
         id              author     subreddit  \
0  18wql7v  Appropriate-Design  ArtistLounge   
1  18x6f70    master6021346666  ArtistLounge   
2  18xj0xd       paint-the-sky  ArtistLounge   
3  18xv1sw         erick_diego  ArtistLounge   
4  18y5pw4           Atomie888  ArtistLounge   

                                               title  \
0             Trying to ID this AI Image's Art-style   
1  Discussion: Anyone else afraid of posting artw...   
2  Is anyone here an art teacher? Im really consi...   
3  Why there isn't a distributor, like Distrokid,...   
4                     What sites do you post art to?   

                                            selftext matched_keywords  
0  Hello all. Recently I was browsing through Pix...               ai  
1  i would love to post and share my artwork onli...               ai  
2  So some background, I have my BFA already whic...               ai  
3  All we have are digital artist platforms like ...               ai  


# Step 2: Campionamento bilanciato per subreddit

Prima di procedere con la classificazione delle opinioni, il dataset dei commenti puliti viene sottoposto a un campionamento bilanciato per subreddit, con l'obiettivo di evitare che le community con più commenti disponibili risultino sovra-rappresentate nelle analisi successive.

## Individuazione del Valore di Bottleneck
Per ciascun subreddit viene contato il numero di commenti disponibili dopo la pulizia. Tra questi conteggi si individua il valore minimo, che costituisce il **valore di bottleneck**: il subreddit con meno commenti determina il numero massimo di commenti utilizzabili per tutti gli altri, poiché non è possibile campionarne di più di quanti effettivamente disponibili.


In [7]:
counts = comments_clean_df.groupby("subreddit").count()
print(counts)
bottleneck_value = counts.min().min()
bottleneck_subreddit = counts.min(axis=1).idxmin()
print(f"\n\nValore di bottleneck: {bottleneck_value} (subreddit: r/{bottleneck_subreddit})")


                   id  link_id  author   body
subreddit                                    
ArtistLounge     8205     8205    8205   8205
AskProgramming   6046     6046    6046   6046
copywriting      2618     2618    2618   2618
graphic_design  13210    13210   13210  13209
marketing        7744     7744    7744   7744
webdev          16768    16768   16768  16764


Valore di bottleneck: 2618 (subreddit: r/copywriting)



## Campionamento Casuale
Una volta stabilito il numero di commenti da estrarre per ciascun subreddit (pari al valore di bottleneck), si procede a un campionamento casuale, senza reinserimento, dei commenti disponibili per ogni subreddit, fino a raggiungere tale numero.

L'estrazione casuale, anziché una selezione basata su un criterio specifico (ad esempio cronologico o per popolarità), garantisce che il sottoinsieme risultante per ciascun subreddit sia rappresentativo della distribuzione complessiva dei commenti di quella community, evitando di introdurre bias sistematici nella selezione.

## Risultato
Al termine di questa fase, ogni subreddit contribuisce al dataset con lo stesso numero di commenti, pari al valore di bottleneck individuato. Questo garantisce che le successive analisi comparative tra professioni (stance analysis, analisi di rete) non risentano di squilibri numerici tra le community, permettendo un confronto equo tra i subreddit considerati.

In [8]:
# Campionamento Casuale

# Seed del Random State fissato per riproducibilità
RANDOM_STATE = 42

# Campionamento casuale di bottleneck_value commenti per ogni subreddit
sampled_comments_df = (
    comments_clean_df
    .groupby("subreddit", group_keys=False)
    .sample(n=bottleneck_value, random_state=RANDOM_STATE, replace=False)
    .reset_index(drop=True)
)

print(f"Commenti campionati: {len(sampled_comments_df):,}")
print(sampled_comments_df.groupby("subreddit").size())

# Salvataggio del campione bilanciato
SAMPLED_COMMENTS_PATH = "../data/all_clean_comments_sampled.csv"
os.makedirs(os.path.dirname(SAMPLED_COMMENTS_PATH), exist_ok=True)
sampled_comments_df.to_csv(SAMPLED_COMMENTS_PATH, index=False, encoding="utf-8")
print(f"\nCampione bilanciato salvato in: {SAMPLED_COMMENTS_PATH}")


Commenti campionati: 15,708
subreddit
ArtistLounge      2618
AskProgramming    2618
copywriting       2618
graphic_design    2618
marketing         2618
webdev            2618
dtype: int64

Campione bilanciato salvato in: ../data/all_clean_comments_sampled.csv


# Step 3: Classificazione delle opinioni

Si effettua una **stance analysis** per ogni utente che ha scritto un commento sotto un post AI-related, in questa maniera si è in grado di fare analisi su reazioni dirette degli utenti che esprimono l'opinione in risposta ad uno stimolo specifico.

### Problema di Role Conflation

Uno stesso utente che appare nel dataset può assumere due ruoli diversi: autore e commentatore del post, questo può causare ambiguità nella rete.

Per risolvere questo problema si applica la seguente strategia:

| Strategia                                | Funzionamento                                                                                                                   |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------- |
| Trattare autore e commenti separatamente | Non si considera il testo del post per la stance e si include l'autore se e solo se ha commentato altri oppure il proprio post. |

In questa maniera il post viene trattato come uno **strumento neutro** che permette la discussione, senza considerarla una vera e propria presa di posizione, d'altro canto i commenti scritti dall'autore sotto il proprio post vengono inclusi nell'analisi esattamente come i commenti di qualsiasi altro utente.
Questo perché un commento viene considerato come **un'azione distinta** dalla pubblicazione del post stesso.
L'autore sta reagendo alla discussione, rispondendo ad altri, elaborando la propria posizione in modo esplicito nei commenti.

Ad ogni utente viene quindi assegnato l'attributo stance che potrà assumere i seguenti valori con i loro rispettivi significati:

|Classe|Significato|
|---|---|
|Pro-AI|AI come opportunità|
|Anti-AI|AI come minaccia|
|Neutral|Opinione non chiara|


## Approccio LLM-based con Gemma 4 E4B
### Motivazione dell'Approccio
Per effettuare la classificazione è stato scelto un approccio **LLM-based zero-shot**: anziché addestrare un classificatore "*supervised*" su dati etichettati manualmente, si sfrutta la capacità di comprensione del linguaggio di un Large Language Model **pre-addestrato**, guidandolo tramite un prompt che descrive il compito e le etichette attese, senza specificare esempi di casi già risolti (zero-shot).

Questo approccio è giustificato da tre considerazioni:

- **Assenza di dati etichettati**: non esiste un dataset di riferimento per la stance verso l'AI su Reddit, rendendo il fine-tuning impraticabile senza un processo di classificazione manuale.
- **Flessibilità semantica**: i commenti Reddit presentano informalità, ironia e spesso ambiguità lessicali, un LLM permette di gestire meglio queste variabilità rispetto a un classificatore tradizionale.
- **Riproducibilità**: l'approccio zero-shot è completamente documentabile tramite il prompt utilizzato, rendendo il processo trasparente e verificabile.

#### Scelta del Modello
Il modello scelto è **Gemma 4 E4B** nella variante **instruction-tuned**[^1] (`google/gemma-4-E4B-it`).

Verrà usata la libreria `transformers` di HuggingFace che consente di scaricare e caricare questi modelli per usarli localmente.
Il modello viene scaricato e salvato nella cache che, per motivi di spazio di archiviazione, viene rediretta impostando la variabile d'ambiente `HF_HOME`, puntando a un percorso su un disco con capacità sufficiente ad archiviare i pesi del modello (circa 3–4GB).

La scelta di Gemma 4 E4B è il risultato di due vincoli:
1. il **vincolo hardware**: il progetto viene eseguito in locale su macchina con un vincolo fisico sulla GPU disponibile
2. il **vincolo di qualità**: viene richiesta una qualità minima per il task di classificazione come la *stance analysis*

Dal punto di vista hardware, la GPU disponibile è una **NVIDIA RTX 2070 Super con 8GB di VRAM**. Questo limite esclude immediatamente i modelli con parametri superiori agli 8B senza quantizzazione aggressiva, che degraderebbe eccessivamente la qualità.

[^1]: La variante **instruction-tuned** (`-it`) è stata scelta deliberatamente rispetto alla versione base: i modelli instruction-tuned sono stati ulteriormente addestrati per seguire istruzioni in linguaggio naturale e produrre risposte strutturate, caratteristica essenziale per ottenere output controllati e parsificabili (es. una singola etichetta testuale) senza post-processing complesso.

#### Installazione del Pacchetto PyTorch
Per installare e avviare correttamente il modello si va uso della libreria pyTorch che permette di effettuare l'offload del modello sulla GPU, permettendone così l'esecizione, che sarebbe altrimenti impossibile su CPU a causa di restrizioni di hardware della macchina su cui viene avviato.
Si fa attenzione quindi a scaricare la versione con accellerazione CUDA, piuttosto che quella solo CPU per poter ottimizzare le prestazioni.


In [3]:
# Setup per Stance Analysis

# Modifica della posizione della cache huggingface
os.environ["HF_HOME"] = "D:\\Models\\huggingface_cache"
print(f"Cache HuggingFace impostata su: {os.environ['HF_HOME']}")

# Import per LLM
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

# ID del modello Gemma 4 E4B
MODEL_ID = "google/gemma-4-E4B-it"

# Login OAuth al hub di huggingface per scaricare il modello
login()


Cache HuggingFace impostata su: D:\Models\huggingface_cache


In [4]:

# Configurazione quantizzazione 4-bit
# - load_in_4bit: riduce i pesi da 32-bit a 4-bit, abbattendo l'uso di VRAM
# - bnb_4bit_compute_dtype: i calcoli interni restano in float16 per stabilità numerica
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Caricamento tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Caricamento modello...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0}   # distribuisce i layer solo su GPU
)
model.eval()  # disabilita dropout e gradient tracking: non stiamo addestrando

print(f"Modello caricato su: {next(model.parameters()).device}")
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Caricamento tokenizer...
Caricamento modello...


W0704 10:41:22.386000 18316 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Modello caricato su: cuda:0
VRAM occupata: 9.53 GB


## Stance Analysis e Inferenza
Si procede con il processo di Stance Analysis dei commenti utilizzando l'inferenza del modello LLM scelto.

Durante la fase di inferenza si prende il modello pre-addestrato (in cui i pesi sono fissi e non vengono aggiornati) e lo si usa per produrre un output a partire da un input.
Durante questa fase quindi non si fa uso né di backpropagation né di calcolo del gradiente, che verranno di conseguenza disattivati con l'obiettivo di risparmiare potenza di calcolo.

## Output Finale
Come output finale della seguente sezioni ci si aspetta un file .csv con la seguente struttura dati:

| Campo  | Spiegazione                           |
|--------|---------------------------------------|
| id     | id del commento analizzato            |
| stance | risultato dell'inferenza sul commento |

Questo file sarà poi utilizzato in combinazione con i dati precedentemente scaturiti dalle fasi 1 e 2 per la corretta analisi ed interpretazione, allo scopo di estrapolare gli insights necessari per rispondere alle domande di ricerca poste.

In [5]:
# Costanti
VALID_LABELS = {"Pro-AI", "Anti-AI", "Neutral"}
FALLBACK_LABEL = "Neutral"  # etichetta assegnata se il modello produce output non parsificabile
MAX_NEW_TOKENS = 10  # il modello deve produrre solo la label, si fissa quindi un numero basso di token

### Scelta e Definizione del Prompt
Viene definito un **system prompt** utilizzando le regole basi del *prompt engineering*:
1. **Assegnazione di un ruolo**:
   Viene attribuito un ruolo specifico, questa tecnica permette di restringere lo spazio delle risposte plausibili e attivare, riducendo la probabilità che il modello produca testo libero invece dell'etichetta richiesta.
2. **Definizione del dominio del task**:
   Viene chiarito l'ambito in cui i termini che deve ricercare (come AI) sono utilizzati, questo viene fatto elencando esempi concreti di strumenti coerenti con le keyword usate per filtrare i dati su cui avverrà l'inferenza.
3. **Definizione operativa per l'etichette di categorizzazione**:
   Viene data una micro-definizione per ogni etichetta, questo permette al modello di rendere il processo di categorizzazione più operazionale piuttosto che un parere più generico.
4. **Vincoli di formato dell'output**:
   Vengono messi dei limiti di modo da migliorare la *parsabilità* dell'output, in questa maniera sarà possibile aumentare la robustezza della risposta, potendo così poi aspettarsi con, maggiore probabilità, la presenza dell'etichetta nell'output.
5. **Assenza di Esempi**:
   Come specificato inizialmente, è stato scelto un approccio *zero-shot*: il prompt non contiene esempi di commenti già classificiati. Questa è una caratteristica di design che permette di rendere più oggettivi i risultati di classificazione effettuati dal modello.
#### Istruction Tuned Prompt
Trattandosi di un modello IT (Istruction Tuned) la struttura del prompt prende un pattern preciso, dovuto alla presenza di ruoli espliciti nella creazione della risposta:
- `system`
- `user`
- `assistant`

Per questo motivo è necessario utilizzare `tokenizer.apply_chat_template(...)` nella costruzione del prompt.
Nei modelli IT il *system prompt* viene tipicamente trattato come un contesto persistente che modula il comportamento in maniera distinta dalla richiesta dell'utente (caratterizzata dai delimitatori `user`).

Questo tipo di modelli sono specificatamente addestrati a seguire istruzioni imperative, ciò permette l'utilizzo di tecniche di classificazione come lo zero-shot in un contesto simile a quello presentato nel seguente progetto.

In [6]:
# Definizione del System Prompt
SYSTEM_PROMPT = """You are a stance classifier. Your task is to classify Reddit comments based on the author's attitude toward AI tools (such as ChatGPT, Copilot, Midjourney, Stable Diffusion, and similar).

Classify each comment into exactly one of these three categories:

- Pro-AI: the author sees AI as an opportunity, expresses enthusiasm, support, or positive outlook
- Anti-AI: the author sees AI as a threat, expresses fear, rejection, or negative outlook
- Neutral: the author's stance is unclear, mixed, or purely informational

Reply with ONLY the label. Do not explain. Do not add punctuation."""


### Generazione controllata
Per assicurarsi il massimo controllo sul risultato viene controllata la generazione della risposta da parte del modello LLM, in maniera particolare si controllano:
- **Greedy Decoding**: la scelta del token finale dalla distribuzione di probabilità generata sarà rappresentata solo dal token con la probabilità più alta, in modo *deterministico*
- **Disattivazione del calcolo del gradiente**: si disattiva il calcolo del gradiente per questioni di efficienza e restrizioni di hardware, trattandosi della fase di inferenza, e non volendo effettuare aggiornamento di pesi, il calcolo del gradiente produrrebbe inutile overhead computazionale.
- **Limitazione dei token di risposta**: viene limitato il numero di token che il modello può generare, questo perché ci si aspetta già una risposta corta (possibilmente solo la label) e si vuole ridurre il più possibile il carico computazionale, di modo da velocizzare l'inferenza ed evitare la saturazione della VRAM con la *KV-cache*
### Parsing del Risultato
Invece di salvare direttamente l'output grezzo, viene cercata la presenza di una delle tre label valide (case-insensitive) nell'output.
Questo approccio permette di standardizzare il risultato e di estrarre una label quando il modello produce una risposta del tipo: "The stance is Neutral" al posto della risposta ideale attesa: "Neutral".
##### Fallback Label
Se nessuna label valida viene trovata nel testo generato, il commento viene etichettato con `FALLBACK_LABEL = "Neutral"` e viene stampato un warning per tracciare questi casi.

In [7]:
# Definizione del metodo per la classificazione

def classify_comment(comment: str) -> str:
    # Definizione dei ruoli (specifico per modelli IT)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": comment[:1024]}
    ]

    # Applicazione del template per modello IT
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to(model.device)

    # Generazione Controllata
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Parsing dell'output
    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(generated, skip_special_tokens=True).strip()

    for label in VALID_LABELS:
        if label.lower() in raw_output.lower():
            return label

    print(f"[WARN] Output non parsificabile: '{raw_output}' → assegnato '{FALLBACK_LABEL}'")
    return FALLBACK_LABEL

In [20]:
# Test su commenti di esempio

# Esempio di commento atteso come "Pro-AI"
test_comment = "AI tools like Copilot are incredible, they doubled my productivity!"
print(f"\n\nTest classificazione: '{test_comment}'")
print(f"Risultato: {classify_comment(test_comment)}")

# Esempio di commento atteso come "Anti-AI"
test_comment = "AI tools like Copilot are awful, they stole my job and my boss thinks they are better than me!"
print(f"\n\nTest classificazione: '{test_comment}'")
print(f"Risultato: {classify_comment(test_comment)}")

# Esempio di commento atteso come "Nutral"
test_comment = "AI tools like Copilot are nice, not good, but just nice, so I see no problem with them"
print(f"\n\nTest classificazione: '{test_comment}'")
print(f"Risultato: {classify_comment(test_comment)}")

Test classificazione: 'AI tools like Copilot are incredible, they doubled my productivity!'


D:\UNI\Corsi\Social Media Mining\Progetto SMM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Risultato: Pro-AI
Test classificazione: 'AI tools like Copilot are awful, they stole my job and my boss thinks they are better than me!'
Risultato: Anti-AI
Test classificazione: 'AI tools like Copilot are nice, not good, but just nice, so I see no problem with them'
Risultato: Neutral


## Inferenza con Checkpoint
Essendo che il processo di analisi di circa 15k commenti tramite un LLM in inferenza locale è un processo che richiede tempi lunghi, per evitare di subire perdita del lavoro effettuato a causa di interruzioni di varia natura si implementa un sistema di checkpoint per il salvataggio periodico del lavoro effettuato.
Questo sistema permette anche di fermare volontariamente l'esecuzione per riprenderla in un secondo momento.

Il sistema di checkpoint funziona tramite i seguenti passaggi:
1. **Setup del punto di Checkpoint**: si controlla se esiste già un file di checkpoint su disco, se si, viene caricato, altrimenti viene creato come `DataFrame` con le colonne `id` e `stance` che verrà poi salvato nell'apposito file di checkpoint .csv
2. **Filtraggio dei Commenti**: il dataset completo viene filtrato per **escludere** tutto ciò che risulta già presente nel checkpoint. Solo i commenti non ancora classificati vengono passati al ciclo di inferenza.
3. **Ciclo di inferenza con Salvataggio Periodico**: i risultati dell'inferenza vengono salvati in un buffer che risiede in memoria centrale, al raggiungimento della soglia, impostata da `CHECKPOINT_EVERY` (nell'attuale caso 500), di commenti il contenuto del buffer viene salvato sul disco nel file di checkpoint e il buffer svuotato.
4. **Flush finale del buffer residuo**: una volta processato l'intero dataset di commenti, l'ultimo buffer viene anch'esso salvato all'interno del checkpoint di modo da assicurarsi che nessun risultato calcolato vada perso
5. **Creazione del file di Output Finale**: il file di checkpoint, che contiene solo `id` e `stance`, viene ricongiunto al dataset completo dei commenti originali tramite `id`, usando una **left join**: ogni riga del dataset originale viene mantenuta, e le viene affiancata la relativa etichetta di stance.

In [9]:
# Setup

from tqdm.notebook import tqdm

# Percorsi file
COMMENTS_PATH    = "../data/all_clean_comments_sampled.csv"      # input: campione bilanciato per subreddit
CHECKPOINT_PATH  = "../data/stanceAnalysis/stanceCheckpoint/vecchio checkpoint (troppi neutral)/stance_checkpoint.csv"  # checkpoint intermedio
OUTPUT_COMMENTS  = "../data/comments_with_stance.csv"     # output finale commenti

# Soglia per salvataggio
CHECKPOINT_EVERY = 500

In [8]:


# Step 1: Setup del punto di Checkpoint
df_comments = pd.read_csv(COMMENTS_PATH)
print(f"Commenti totali: {len(df_comments):,}")

# Riprende da checkpoint se esiste
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    already_done  = set(df_checkpoint["id"])
    print(f"Checkpoint trovato: {len(already_done):,} commenti già classificati, riprendo da lì.")
else:
    df_checkpoint = pd.DataFrame(columns=["id", "stance"])
    already_done  = set()
    print("Nessun checkpoint trovato, parto dall'inizio.")

# Step 2: Filtraggio dei Commenti
df_todo = df_comments[~df_comments["id"].isin(already_done)].reset_index(drop=True)
print(f"Commenti da classificare: {len(df_todo):,}")

# Step 3: Ciclo di inferenza con Salvataggio Periodico
results = []  # buffer temporaneo tra un checkpoint e l'altro

for i, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Classificazione"):
    stance = classify_comment(row["body"])
    results.append({"id": row["id"], "stance": stance})

    # Salvataggio checkpoint periodico
    if (i + 1) % CHECKPOINT_EVERY == 0:
        df_new        = pd.DataFrame(results)
        df_checkpoint = pd.concat([df_checkpoint, df_new], ignore_index=True)
        df_checkpoint.to_csv(CHECKPOINT_PATH, index=False)
        results = []  # svuota il buffer
        print(f"  Checkpoint salvato: {len(df_checkpoint):,} commenti totali processati")

# Step 4: Flush finale del buffer residuo
if results:
    df_new        = pd.DataFrame(results)
    df_checkpoint = pd.concat([df_checkpoint, df_new], ignore_index=True)
    df_checkpoint.to_csv(CHECKPOINT_PATH, index=False)

# Step 5: Creazione del file di Output Finale
df_output = df_comments.merge(df_checkpoint[["id", "stance"]], on="id", how="left")
df_output.to_csv(OUTPUT_COMMENTS, index=False)

print(f"\nInferenza completata. Output salvato in '{OUTPUT_COMMENTS}'")
print(df_output["stance"].value_counts())

Commenti totali: 15,708
Checkpoint trovato: 8,500 commenti già classificati, riprendo da lì.
Commenti da classificare: 7,208


Classificazione:   0%|          | 0/7208 [00:00<?, ?it/s]

D:\UNI\Corsi\Social Media Mining\Progetto SMM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[WARN] Output non parsificabile: 'Mixed' → assegnato 'Neutral'
  Checkpoint salvato: 9,000 commenti totali processati


D:\UNI\Corsi\Social Media Mining\Progetto SMM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

  Checkpoint salvato: 15,000 commenti totali processati


D:\UNI\Corsi\Social Media Mining\Progetto SMM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Checkpoint salvato: 15,500 commenti totali processati


D:\UNI\Corsi\Social Media Mining\Progetto SMM\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Inferenza completata. Output salvato in '../data/comments_with_stance.csv'
stance
Neutral    11565
Pro-AI      2139
Anti-AI     2004
Name: count, dtype: int64


In [32]:
df_output = pd.read_csv(OUTPUT_COMMENTS)
df_output



,id,link_id,subreddit,author,body,stance
0,kqla5k2,t3_1ap0cm3,ArtistLounge,TwoRemote8471,I saw a few comments from other sub Reddit’s t...,Neutral
1,l56y343,t3_1cy37zl,ArtistLounge,Background_City_8575,"It sucks. I wish deviantart made a comeback, o...",Anti-AI
2,m3x6dmp,t3_1hmclcs,ArtistLounge,EricYoungArt,The sad truth is AI killed digital painting fo...,Anti-AI
3,m3vqruq,t3_1hmclcs,ArtistLounge,de4dite,Years ago when I first switched over to digita...,Neutral
4,kphk0vp,t3_1alvy9h,ArtistLounge,NeonFraction,"Hi, full time 3D artist here who has also hire...",Neutral
...,...,...,...,...,...,...
15703,l25ecuc,t3_1cgszay,webdev,I_will_delete_myself,"Good tech, Over washed from just a simple feat...",Neutral
15704,kunlez8,t3_1bdmv02,webdev,Particular_Pudding81,"When I develop new features in my code, very o...",Neutral
15705,l8ophtg,t3_1dfslqq,webdev,Longjumping_Car6891,It's less about understanding code; understand...,Neutral
15706,ktr1erf,t3_1b8poar,webdev,TTuserr,beside PHP you will also need to know WP stuff...,Neutral


# Step 4: Validazione Umana della Classificazione

La classificazione ottenuta tramite il modello LLM mostra una quota di commenti "Neutral" particolarmente elevata (73,6% del totale). Per verificare che questo risultato rifletta un pattern reale nei dati, e non un bias del modello (es. tendenza a "rifugiarsi" nella classe neutra quando incerto, oppure a considerare "Neutral" anche commenti informativi o off-topic che non esprimono alcuna stance), si effettua una validazione umana su un campione di commenti.

La validazione si articola in tre fasi:
1. **Campionamento**: si estrae un campione stratificato dal bucket "Neutral" (per diagnosticarne la composizione interna) piu' un campione piu' piccolo di "Pro-AI" e "Anti-AI" (per rigore, cioe' per verificare che l'accordo modello-umano non sia un problema specifico della sola classe neutra).
2. **Annotazione manuale in cieco**: l'etichetta assegnata dal modello non viene mostrata durante l'annotazione, per non influenzare il giudizio dell'annotatore umano. Ogni commento annotato come "Neutral" richiede anche di specificarne il motivo: neutralita' genuina, off-topic (il commento non riguarda l'AI) o puramente informativo (nessuna stance espressa).
3. **Analisi**: si calcola la composizione interna del bucket neutro secondo l'annotazione umana e si misura l'accordo modello-umano tramite accuracy e Cohen's kappa.

In [33]:
# Costanti per il campionamento del set di validazione

NEUTRAL_SAMPLE_SIZE     = 180
PRO_SAMPLE_SIZE         = 50
ANTI_SAMPLE_SIZE        = 50
VALIDATION_RANDOM_STATE = 42

VALIDATION_DIR              = Path("../data/stanceAnalysis/humanValidation")
VALIDATION_SAMPLE_PATH      = VALIDATION_DIR / "validation_sample.csv"
VALIDATION_ANNOTATIONS_PATH = VALIDATION_DIR / "validation_annotations.csv"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# Il campionamento viene eseguito una sola volta: se il file esiste gia', viene
# ricaricato cosi' com'e', per non invalidare un'annotazione manuale gia' in corso
if VALIDATION_SAMPLE_PATH.exists():
    validation_sample_df = pd.read_csv(VALIDATION_SAMPLE_PATH)
    print(f"Campione di validazione gia' esistente, caricato da '{VALIDATION_SAMPLE_PATH}'.")
else:
    neutral_sample = df_output[df_output["stance"] == "Neutral"].sample(
        n=NEUTRAL_SAMPLE_SIZE, random_state=VALIDATION_RANDOM_STATE
    )
    pro_sample = df_output[df_output["stance"] == "Pro-AI"].sample(
        n=PRO_SAMPLE_SIZE, random_state=VALIDATION_RANDOM_STATE
    )
    anti_sample = df_output[df_output["stance"] == "Anti-AI"].sample(
        n=ANTI_SAMPLE_SIZE, random_state=VALIDATION_RANDOM_STATE
    )

    # Le righe vengono mischiate cosi' che l'ordine di apparizione non riveli
    # la stance assegnata dal modello durante l'annotazione manuale
    validation_sample_df = (
        pd.concat([neutral_sample, pro_sample, anti_sample], ignore_index=True)
        .sample(frac=1, random_state=VALIDATION_RANDOM_STATE)
        .reset_index(drop=True)
    )
    validation_sample_df.to_csv(VALIDATION_SAMPLE_PATH, index=False)
    print(f"Campione di validazione creato e salvato in '{VALIDATION_SAMPLE_PATH}'.")

print(f"\nCampione totale: {len(validation_sample_df)} commenti")
print(validation_sample_df["stance"].value_counts())

Campione di validazione creato e salvato in '..\data\stanceAnalysis\humanValidation\validation_sample.csv'.

Campione totale: 280 commenti
stance
Neutral    180
Anti-AI     50
Pro-AI      50
Name: count, dtype: int64


## Annotazione Manuale

Eseguire la cella seguente per avviare (o riprendere) l'annotazione manuale. Per ogni commento mostrato, digitare:
- `p` se il commento e' **Pro-AI**
- `a` se il commento e' **Anti-AI**
- `n` se il commento e' **Neutral** (verra' poi richiesto il motivo: genuino / off-topic / informativo)
- `s` per saltare il commento (restera' da annotare)
- `q` per interrompere e salvare il lavoro fatto finora (si potra' riprendere rieseguendo la cella)

Le annotazioni vengono salvate progressivamente su file: l'annotazione puo' quindi essere interrotta e ripresa in piu' sessioni senza perdere il lavoro gia' fatto.

In [35]:
# Etichette e motivazioni utilizzabili durante l'annotazione manuale
HUMAN_STANCE_KEYS   = {"p": "Pro-AI", "a": "Anti-AI", "n": "Neutral"}
NEUTRAL_REASON_KEYS = {"g": "Genuine", "o": "Off-topic", "i": "Informational"}


def _load_annotations() -> pd.DataFrame:
    if VALIDATION_ANNOTATIONS_PATH.exists():
        return pd.read_csv(VALIDATION_ANNOTATIONS_PATH)
    return pd.DataFrame(columns=["id", "human_stance", "human_neutral_reason"])


def annotate_validation_sample(sample_df: pd.DataFrame) -> pd.DataFrame:
    """
    Annotazione manuale "in cieco": mostra solo testo e subreddit del commento,
    MAI l'etichetta assegnata dal modello, per non influenzare il giudizio
    dell'annotatore. Se la stance scelta e' "Neutral", chiede di specificarne
    anche il motivo (genuino / off-topic / informativo), utile per scomporre
    il bucket neutro nell'analisi successiva.

    Le annotazioni vengono salvate progressivamente su file: l'annotazione puo'
    essere interrotta in qualsiasi momento (comando 'q') e ripresa in seguito
    rieseguendo la funzione, che salta i commenti gia' annotati.
    """
    annotations_df = _load_annotations()
    already_annotated = set(annotations_df["id"])
    todo_df = sample_df[~sample_df["id"].isin(already_annotated)].reset_index(drop=True)

    print(f"Commenti gia' annotati: {len(already_annotated)}/{len(sample_df)}")
    print("Comandi: [p]ro-AI  [a]nti-AI  [n]eutral  [s]kip  [q]uit\n")

    for i, row in todo_df.iterrows():
        print(f"\n[{len(already_annotated) + i + 1}/{len(sample_df)}] r/{row['subreddit']}")
        print(f"\"{row['body']}\"")

        choice = input("Stance percepita [p/a/n/s/q]: ").strip().lower()

        if choice == "q":
            print("\nAnnotazione interrotta. Lavoro salvato, si puo' riprendere in seguito.")
            break
        if choice not in HUMAN_STANCE_KEYS:
            continue  # 's' (skip) o input non valido: il commento resta da annotare

        human_stance = HUMAN_STANCE_KEYS[choice]
        neutral_reason = None

        if human_stance == "Neutral":
            reason_choice = input(
                "Motivo della neutralita' [g]enuina / [o]fftopic / [i]nformativo: "
            ).strip().lower()
            neutral_reason = NEUTRAL_REASON_KEYS.get(reason_choice, "Genuine")

        new_row = pd.DataFrame([{
            "id": row["id"],
            "human_stance": human_stance,
            "human_neutral_reason": neutral_reason,
        }])
        annotations_df = pd.concat([annotations_df, new_row], ignore_index=True)
        annotations_df.to_csv(VALIDATION_ANNOTATIONS_PATH, index=False)  # salvataggio progressivo

    return annotations_df


annotations_df = annotate_validation_sample(validation_sample_df)
print(f"\nTotale annotazioni salvate: {len(annotations_df)}/{len(validation_sample_df)}")

Commenti gia' annotati: 0/280
Comandi: [p]ro-AI  [a]nti-AI  [n]eutral  [s]kip  [q]uit


[1/280] r/AskProgramming
"Because we are very far from superintelligence"

[2/280] r/AskProgramming
"So full stack web dev will never be “saturated”, however the software developer market follows the interest rates of money closely. Software companies generally either lose money or make a lot of money, so they are risky investments. When money is cheap and people are willing to gamble on new software companies there are a lot more jobs. Currently interest rates are high, which is a big reason you are not seeing a lot of full stack jobs."

[3/280] r/graphic_design
"This is a clear indication that the "designer" had no idea what they were doing and has no business selling any sort of design services, AI-generated or not."

[4/280] r/graphic_design
"Never. This is something which should have been dealt with at the beginning of the process - choosing the right photography - not bodging faces at the end 

KeyboardInterrupt: Interrupted by user

## Analisi: Composizione del Bucket Neutral e Accordo Modello-Umano

**Nota metodologica:** poiche' il campione e' stratificato sull'etichetta assegnata dal modello (non estratto uniformemente dall'intero dataset), l'accuracy calcolata qui non equivale all'accuracy complessiva del modello sull'intero dataset, ma misura, per ciascuna classe predetta, quanto spesso l'annotatore umano conferma l'etichetta del modello (un indicatore analogo alla *precision* per classe). Cohen's kappa fornisce comunque una misura dell'accordo corretta per il caso, utile per capire se il modello sta operando scelte sistematiche o sostanzialmente casuali.

In [ ]:
import numpy as np

# Unione tra campione, etichetta del modello e annotazione umana
validation_results_df = validation_sample_df.merge(annotations_df, on="id", how="inner")

missing = len(validation_sample_df) - len(validation_results_df)
if missing:
    print(f"[WARN] {missing} commenti del campione non ancora annotati: esclusi dall'analisi.\n")

# Composizione del bucket "Neutral" secondo l'annotazione umana:
# se l'annotatore conferma "Neutral", si usa il motivo (Genuine/Off-topic/Informational);
# se l'annotatore lo ha invece classificato come Pro-AI o Anti-AI, il modello ha sbagliato stance
neutral_bucket = validation_results_df[validation_results_df["stance"] == "Neutral"].copy()
neutral_bucket["human_category"] = neutral_bucket["human_neutral_reason"].fillna(neutral_bucket["human_stance"])

neutral_composition     = neutral_bucket["human_category"].value_counts()
neutral_composition_pct = (neutral_composition / len(neutral_bucket) * 100).round(1)

print(f"Composizione del bucket 'Neutral' secondo annotazione umana (n={len(neutral_bucket)}):")
for label, count in neutral_composition.items():
    print(f"  {label:<15} {neutral_composition_pct[label]:>5.1f}%  ({count})")


def cohens_kappa(rater1: pd.Series, rater2: pd.Series, labels: list) -> float:
    """
    Calcola l'agreement di Cohen's kappa tra due annotatori (o annotatore e
    modello), corretto per l'accordo atteso per puro caso:
    kappa = (osservato - atteso) / (1 - atteso)
    """
    n = len(rater1)
    confusion = pd.crosstab(rater1, rater2).reindex(index=labels, columns=labels, fill_value=0)

    observed_agreement = np.trace(confusion.values) / n
    row_marginals = confusion.sum(axis=1) / n
    col_marginals = confusion.sum(axis=0) / n
    expected_agreement = (row_marginals * col_marginals).sum()

    return (observed_agreement - expected_agreement) / (1 - expected_agreement)


STANCE_LABELS = ["Pro-AI", "Anti-AI", "Neutral"]
y_model = validation_results_df["stance"]
y_human = validation_results_df["human_stance"]

accuracy = (y_human == y_model).mean()
kappa    = cohens_kappa(y_human, y_model, STANCE_LABELS)

print(f"\nAccordo modello vs annotazione umana (n={len(validation_results_df)}):")
print(f"  Accuracy:      {accuracy:.3f}")
print(f"  Cohen's kappa: {kappa:.3f}")

print("\nPrecisione del modello per classe predetta (su questo campione stratificato):")
for label in STANCE_LABELS:
    subset = validation_results_df[validation_results_df["stance"] == label]
    if len(subset):
        match_rate = (subset["human_stance"] == label).mean()
        print(f"  {label:<8} predetto {len(subset):>3} volte -> confermato dall'annotatore umano nel {match_rate:.1%} dei casi")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Colori categoriali: assegnazione fissa per categoria, non per rango/valore
CATEGORY_COLORS = {
    "Genuine":       "#2a78d6",  # blue
    "Off-topic":     "#1baf7a",  # aqua
    "Informational": "#eda100",  # yellow
    "Pro-AI":        "#008300",  # green
    "Anti-AI":       "#e34948",  # red
}
MUTED_GRID  = "#e1e0d9"
MUTED_TEXT  = "#898781"
PRIMARY_INK = "#0b0b0b"

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Composizione del bucket Neutral ---
ax = axes[0]
composition_sorted = neutral_composition_pct.sort_values(ascending=True)
bar_colors = [CATEGORY_COLORS.get(label, MUTED_TEXT) for label in composition_sorted.index]

bars = ax.barh(composition_sorted.index, composition_sorted.values, color=bar_colors, height=0.6)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in composition_sorted.values], padding=4, color=PRIMARY_INK, fontsize=9)
ax.set_xlabel("% del bucket 'Neutral'", color=MUTED_TEXT)
ax.set_title(f"Composizione del bucket Neutral (n={len(neutral_bucket)})", color=PRIMARY_INK, loc="left")
ax.set_xlim(0, max(composition_sorted.values) * 1.25)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("left", "bottom"):
    ax.spines[spine].set_color(MUTED_GRID)
ax.tick_params(colors=MUTED_TEXT)
ax.xaxis.grid(True, color=MUTED_GRID, linewidth=0.8)
ax.set_axisbelow(True)

# --- Matrice di confusione modello vs annotazione umana ---
ax = axes[1]
confusion = pd.crosstab(y_human, y_model).reindex(index=STANCE_LABELS, columns=STANCE_LABELS, fill_value=0)
seq_blue = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)

im = ax.imshow(confusion.values, cmap=seq_blue)
ax.set_xticks(range(len(STANCE_LABELS)), labels=STANCE_LABELS, color=MUTED_TEXT)
ax.set_yticks(range(len(STANCE_LABELS)), labels=STANCE_LABELS, color=MUTED_TEXT)
ax.set_xlabel("Etichetta del modello (LLM)", color=MUTED_TEXT)
ax.set_ylabel("Etichetta umana (ground truth)", color=MUTED_TEXT)
ax.set_title(f"Accordo modello-umano (acc={accuracy:.2f}, kappa={kappa:.2f})", color=PRIMARY_INK, loc="left")

for i in range(len(STANCE_LABELS)):
    for j in range(len(STANCE_LABELS)):
        value = confusion.values[i, j]
        text_color = "white" if value > confusion.values.max() / 2 else PRIMARY_INK
        ax.text(j, i, str(value), ha="center", va="center", color=text_color)

plt.tight_layout()
plt.show()